# OCR Extraction Pipeline — Setup & Test

Full litmus test for the extraction pipeline. Run this after:
1. Starting vLLM in a Jupyter terminal
2. Uploading sample docs to `/ocr/`

This notebook will:
- Verify vLLM is running
- Check GPU availability
- Load a sample document and check if it's digital or scanned
- Run extraction (text path or VLM path)
- Inspect the JSON output
- Let you experiment with different formats

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

## 2. Verify vLLM is running

You should have started vLLM in a Jupyter terminal:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-VL-7B-Instruct \
    --dtype auto \
    --max-model-len 8192 \
    --limit-mm-per-prompt image=1
```

In [ ]:
import httpx
import os

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8000/v1")

try:
    resp = httpx.get(f"{LLM_BASE_URL}/models", timeout=10.0)
    models = resp.json()
    MODEL_ID = models["data"][0]["id"]
    print(f"vLLM is running. Model: {MODEL_ID}")
except Exception as e:
    print(f"ERROR: Cannot reach vLLM at {LLM_BASE_URL}")
    print(f"Did you start it in a terminal? Error: {e}")

## 3. Check shared models PVC

In [ ]:
from pathlib import Path

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")

## 4. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload
button, then set the path below.

In [ ]:
# List what's in /ocr
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print(f"Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
# Set this to one of your uploaded files
DOC_PATH = Path("/ocr/sample.pdf")  # <-- UPDATE THIS

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 5. Check digital vs scanned

Try to extract text from each page. If a page has extractable text,
it's digital (fast path). If not, it's scanned (VLM path).

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_info = []
for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    page_info.append({"page": i, "text": text, "has_text": has_text})
    status = "DIGITAL" if has_text else "SCANNED"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:150]}...")
    print()

digital = sum(1 for p in page_info if p["has_text"])
scanned = sum(1 for p in page_info if not p["has_text"])
print(f"Summary: {digital} digital, {scanned} scanned")
doc.close()

## 6. Run extraction

Pick a page and run the appropriate extraction path.

In [ ]:
import time
import base64
import io
from PIL import Image

# Which page to test (0-indexed)
PAGE_IDX = 0
info = page_info[PAGE_IDX]

# Prompt — try different ones!
PROMPT = """Extract all information from this document.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

t0 = time.time()

if info["has_text"]:
    # DIGITAL PATH — send extracted text to LLM
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path (text extraction + LLM)\n")
    full_prompt = f"{PROMPT}\n\n---\nDOCUMENT TEXT:\n---\n{info['text']}"
    resp = httpx.post(
        f"{LLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL_ID,
            "messages": [{"role": "user", "content": full_prompt}],
            "max_tokens": 4096,
            "temperature": 0.0,
        },
        timeout=120.0,
    )
else:
    # SCANNED PATH — render page as image, send to VLM
    print(f"Page {PAGE_IDX+1}: Using SCANNED path (VLM OCR)\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    
    resp = httpx.post(
        f"{LLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL_ID,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": PROMPT},
                ],
            }],
            "max_tokens": 4096,
            "temperature": 0.0,
        },
        timeout=120.0,
    )

elapsed = time.time() - t0
result = resp.json()["choices"][0]["message"]["content"]
print(f"\nExtraction took {elapsed:.1f}s")

## 7. Inspect the output

In [ ]:
import json

try:
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Not valid JSON: {e}\n")
    print("Raw output:")
    print(result)

## 8. Try different prompts

Edit the `PROMPT` variable in step 6 and re-run. Some options:

In [ ]:
# Key-value pairs — more flexible, works on any document
PROMPT_KV = """Extract all labeled data points from this document as key-value pairs.
Return a JSON object where keys are the field names and values are their values.
Preserve ALL values exactly. Output only valid JSON."""

# Budget
PROMPT_BUDGET = """Extract budget information from this document.
Return a JSON object with: award_number, budget_period,
categories (array of {category, items: [{description, amount}], subtotal}),
total_direct, fa_rate, fa_base, total_indirect, total, cost_sharing, notes.
Preserve ALL dollar amounts exactly. Output only valid JSON."""

# Terms & conditions
PROMPT_TERMS = """Extract terms and conditions from this document.
Return a JSON object with: document_title, effective_date,
sections (array of {number, title, text, subsections}),
definitions, references.
Preserve exact wording. Output only valid JSON."""

# Raw text — see what the VLM actually reads
PROMPT_TEXT = """Extract all text from this document exactly as it appears.
Preserve the original reading order, line breaks, and structure.
Output only the extracted text."""

print("Copy one of these into the PROMPT variable in step 6 and re-run.")
print("Available: PROMPT_KV, PROMPT_BUDGET, PROMPT_TERMS, PROMPT_TEXT")

## 9. Process all pages

Once you're happy with a prompt, run it on all pages of the document.

In [ ]:
results = []
doc = fitz.open(str(DOC_PATH))

for info in page_info:
    t0 = time.time()
    
    if info["has_text"]:
        full_prompt = f"{PROMPT}\n\n---\nDOCUMENT TEXT:\n---\n{info['text']}"
        resp = httpx.post(
            f"{LLM_BASE_URL}/chat/completions",
            json={"model": MODEL_ID, "messages": [{"role": "user", "content": full_prompt}],
                  "max_tokens": 4096, "temperature": 0.0},
            timeout=120.0,
        )
        method = "text_extraction"
    else:
        mat = fitz.Matrix(2.0, 2.0)
        pix = doc[info["page"]].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        resp = httpx.post(
            f"{LLM_BASE_URL}/chat/completions",
            json={"model": MODEL_ID, "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": PROMPT},
            ]}], "max_tokens": 4096, "temperature": 0.0},
            timeout=120.0,
        )
        method = "vlm_ocr"
    
    elapsed = time.time() - t0
    text = resp.json()["choices"][0]["message"]["content"]
    results.append({"page": info["page"] + 1, "method": method,
                    "elapsed_ms": round(elapsed * 1000, 1), "text": text})
    print(f"Page {info['page']+1}: {method} ({elapsed:.1f}s)")

doc.close()
print(f"\nDone. {len(results)} pages processed.")

In [ ]:
# Save results as JSON
output = {
    "source_file": str(DOC_PATH),
    "total_pages": len(results),
    "digital_pages": sum(1 for r in results if r["method"] == "text_extraction"),
    "scanned_pages": sum(1 for r in results if r["method"] == "vlm_ocr"),
    "pages": results,
}

out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.json")
out_path.write_text(json.dumps(output, indent=2))
print(f"Saved to {out_path}")

## Next steps

Once you're happy with the output:
- **Batch processing:** Use `batch_extract.py` to process many docs at once (see Step 4 in deployment guide)
- **Streamlit app:** Test the interactive UI from this workspace (see Step 4 in setup doc)
- **Deploy:** Set up persistent vLLM endpoint + batch workspace for production